In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import re

In [14]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/CHATMODEL/lora_adapter"

In [15]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [19]:
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)
model.eval()

ValueError: You are trying to offload the whole model to the disk. Please use the `disk_offload` function instead.

In [7]:
SYSTEM_PROMPT = """
أنت مساعد طبي للمساعدة الأولية فقط داخل تطبيق صحي.
افهم أي لغة، لكن رد بالعربية دائمًا.
قواعد صارمة:
1) لا علاج/لا أدوية/لا جرعات.
2) لا تشخيص نهائي.
3) اذكر 2 إلى 4 أمراض محتملة فقط.
4) اختر الأمراض فقط من قائمة "الأمراض المرشحة" ولا تذكر أي مرض خارجها.
5) إذا المعلومات غير كافية: اسأل سؤالين توضيحيين محددين.
6) إذا ظهرت مؤشرات خطيرة (ضيق تنفس شديد/ألم صدر شديد/إغماء/شلل/نزيف شديد): وجّه للطوارئ فورًا.
7) استخدم اللغة العربية فقط في جميع أجزاء الرد.
صيغة الرد (التزم بها حرفيًا):
🔎 تحليل الأعراض:
🦠 أمراض محتملة:
❓ معلومات إضافية قد تساعد:
⚠️ تنبيه:
"""

In [12]:
def suggest_specialty_from_symptoms(symptoms_text: str) -> str:
    text = " " + symptoms_text.lower().strip() + " "  # عشان نضمن حدود الكلمات

    if any(kw in text for kw in [
        "طفح", "حكة", "احمرار", "بثور", "الجلد", "rash", "itch", "skin", "حساسية جلد", "جلدية",
        "dryness", "pimple", "spots", "warts", "eczema", "اكزيما", "حب شباب", "قشرة", "فطريات"
    ]):
        return "أمراض جلدية"

    if any(kw in text for kw in [
        "سعال", "كحة", "ضيق تنفس", "ألم صدر", "تنفس", "breath", "cough", "chest", "wheezing",
        "asthma", "phlegm", "shortness of breath", "بلغم", "نهجان", "حساسية صدر", "ربو"
    ]):
        return "أمراض صدرية وجهاز تنفسي"

    if any(kw in text for kw in [
        "gum", "jaw", "tooth", "sensitivity", "facial swelling", "toothache", "الأسنان",
        "تسوس ", "اللثة", "أسنان المكسورة", "cavity", "bleeding gums", "wisdom tooth", "دروس", "فكي", "خلع"
    ]):
        return "فم وأسنان"

    if any(kw in text for kw in [
        "صداع", "دوخة", "غثيان مع صداع", "حساسية ضوء", "migraine", "headache", "شقيقة", "دوار",
        "numbness", "seizures", "tremors", "تنميل", "تشنجات", "فقدان توازن", "صرع", "اعصاب"
    ]):
        return "مخ وأعصاب"

    if any(kw in text for kw in [
        "بطن", "معدة", "إسهال", "قيء", "غثيان", "ألم أسفل يمين البطن", "abdomen", "stomach", "diarrhea",
        "bloating", "constipation", "acid reflux", "heartburn", "قولون", "امساك", "حموضة", "انتفاخ"
    ]):
        return "جهاز هضمي"

    if any(kw in text for kw in [
        "بول", "حرقان", "كثرة تبول", "ألم أسفل بطن مع تبول", "urine", "burning urination",
        "kidney stone", "bladder", "prostate", "كلى", "مثانة", "حصوات", "بروستاتا", "تبول"
    ]):
        return "مسالك بولية"

    if any(kw in text for kw in [
        "مفاصل", "ركبة", "عظام", "تورم", "joint", "knee", "back pain", "عضم", "fracture",
        "arthritis", "spine", "neck pain", "غضروف", "ديسك", "فقرات", "كسر", "خشونة"
    ]):
        return "عظام ومفاصل"

    if any(kw in text for kw in [
        "قلب", "خفقان", "ألم صدر", "ضيق تنفس مع خوف", "heart", "palpitation", "blood pressure",
        "arrhythmia", "pulse", "ضغط", "نبض", "شرايين", "نهجان مع مجهود"
    ]):
        return "قلب وأوعية دموية"

    if any(kw in text for kw in [
        "حمل", "دورة", "نزيف", "ألم حوض", "نساء", "pregnancy", "menstrual", "period",
        "ovary", "uterus", "infertility", "تأخر حمل", "مبايض", "رحم", "اجهاض"
    ]):
        return "نساء وتوليد"

    if any(kw in text for kw in [
        "عين", "رؤية", "زغلولة", "احمرار العين", "نظر", "eye", "vision", "blurred", "redness", "conjunctivitis", "رمد"
    ]):
        return "رمد وعيون"

    if any(kw in text for kw in [
        "أذن", "حلق", "أنف", "جيوب أنفية", "احتقان", "ear", "throat", "nose", "sinus", "hearing", "طنين", "صعوبة بلع"
    ]):
        return "أنف وأذن وحنجرة"

    if any(kw in text for kw in [
        "اكتئاب", "قلق", "توتر", "نوم", "وسواس", "depression", "anxiety", "insomnia", "stress", "mood", "نوبات هلع"
    ]):
        return "صحة نفسية"

    # لو مفيش تطابق
    return "باطنة"

In [1]:
def ask_medical_ai(symptoms: str, max_new_tokens=220, temperature=0.5):
    user_text = f"""الأعراض: {symptoms.strip()}
    المطلوب:
- فكر في الأعراض فقط ولا تذكر أي قائمة مرشحة أو أمراض محددة من تدريب سابق.
- اقترح 2-4 أمراض شائعة أو محتملة منطقيًا جدًا مع الأعراض دي.
- اذكر سبب مختصر لكل مرض (ليه يناسب الأعراض؟).
- التزم تمامًا بالصيغة: 🔎 🦠 ❓ ⚠️
- ممنوع ذكر أي علاج أو دواء أو جرعة."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.92,
            repetition_penalty=1.08,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # تنظيف (غالباً يبدأ بعد كلمة assistant)
    if "assistant" in response:
        response = response.split("assistant", 1)[-1].strip()

    # استخراج واقتراح التخصص
    specialty = suggest_specialty_from_symptoms(symptoms);

    disease_translation = {
    "abdominal aortic aneurysm": "تمدد الشريان الأورطي البطني","abdominal hernia": "فتق بطني","abscess of nose": "خراج الأنف","abscess of the lung": "خراج الرئة","abscess of the pharynx": "خراج البلعوم",
    "achalasia": "تعذر الارتخاء المريئي","acne": "حبّ الشباب","actinic keratosis": "التقرّن الشعاعي","acute bronchiolitis": "التهاب القصيبات الحاد","acute bronchitis": "التهاب الشعب الهوائية الحاد","acute bronchospasm": "تشنّج قصبي حاد",
    "acute fatty liver of pregnancy (aflp)": "الكبد الدهني الحاد أثناء الحمل","acute glaucoma": "زَرَق حاد (مياه زرقاء حادة)","acute kidney injury": "إصابة/فشل كلوي حاد","acute otitis media": "التهاب الأذن الوسطى الحاد","acute pancreatitis": "التهاب البنكرياس الحاد",
    "acute respiratory distress syndrome (ards)": "متلازمة الضائقة التنفسية الحادة","acute sinusitis": "التهاب الجيوب الأنفية الحاد",
    "acute stress reaction": "استجابة/تفاعل الضغط النفسي الحاد","adhesive capsulitis of the shoulder": "التهاب المحفظة اللاصق بالكتف (الكتف المتجمدة)","adjustment reaction": "اضطراب التكيّف","adrenal adenoma": "ورم غُدي كظري","adrenal cancer": "سرطان الغدة الكظرية","alcohol abuse": "إساءة استخدام الكحول","alcohol intoxication": "تسمّم كحولي","alcohol withdrawal": "أعراض انسحاب الكحول",
    "alcoholic liver disease": "مرض الكبد الكحولي","allergy": "حساسية","allergy to animals": "حساسية الحيوانات","alopecia": "الثعلبة (تساقط الشعر)","alzheimer disease": "مرض ألزهايمر","amblyopia": "كسل العين","amyloidosis": "الداء النشواني",
    "amyotrophic lateral sclerosis (als)": "التصلّب الجانبي الضموري","anal fissure": "شرخ شرجي","anal fistula": "ناسور شرجي","anemia": "فقر الدم","anemia due to chronic kidney disease": "فقر دم بسبب مرض كلوي مزمن","anemia due to malignancy": "فقر دم بسبب خباثة/سرطان","anemia of chronic disease": "فقر الدم المصاحب للأمراض المزمنة","angina": "ذبحة صدرية","ankylosing spondylitis": "التهاب الفقار اللاصق",
    "anxiety": "قلق","aortic valve disease": "مرض/اعتلال الصمام الأورطي","aphakia": "انعدام العدسة","aphthous ulcer": "قرحة فموية قلاعية","aplastic anemia": "فقر الدم اللاتنسّجي","appendicitis": "التهاب الزائدة الدودية","arrhythmia": "اضطراب نظم القلب","arthritis of the hip": "التهاب مفصل الورك",
    "ascending cholangitis": "التهاب الأقنية الصفراوية الصاعد","asperger syndrome": "متلازمة أسبرجر","asthma": "ربو","astigmatism": "استجماتيزم (لابؤرية)","atelectasis": "انخماص الرئة","athlete's foot": "سعفة القدم (فطريات القدم)","atonic bladder": "مثانة رخوة/وهنية","atrial fibrillation": "رجفان أذيني","atrial flutter": "رفرفة أذينية","atrophic skin condition": "حالة ضمور جلدي",
    "atrophic vaginitis": "التهاب المهبل الضموري","attention deficit hyperactivity disorder (adhd)": "اضطراب فرط الحركة وتشتت الانتباه","autism": "التوحّد","autonomic nervous system disorder": "اضطراب الجهاز العصبي الذاتي","balanitis": "التهاب الحشفة","bell palsy": "شلل بيل (شلل العصب الوجهي)","benign kidney cyst": "كيس كلوي حميد",
    "benign paroxysmal positional vertical (bppv)": "دوار الوضعة الانتيابي الحميد",
    "benign prostatic hyperplasia (bph)": "تضخّم البروستاتا الحميد",
    "benign vaginal discharge (leukorrhea)": "إفرازات مهبلية حميدة (سَيْلان/بيض المهبل)",
    "bipolar disorder": "اضطراب ثنائي القطب",
    "bladder cancer": "سرطان المثانة",
    "bladder disorder": "اضطراب المثانة",
    "bladder obstruction": "انسداد المثانة",
    "blepharitis": "التهاب جفن العين",
    "bone cancer": "سرطان العظام",
    "bone disorder": "اضطراب العظام",
    "bone spur of the calcaneous": "شوكة عظمية بالكعب",
    "brachial neuritis": "التهاب العصب العضدي",
    "brain cancer": "سرطان الدماغ",
    "breast cancer": "سرطان الثدي",
    "breast cyst": "كيس بالثدي",
    "breast infection (mastitis)": "التهاب الثدي",
    "broken tooth": "كسر بالسن",
    "burn": "حرق",
    "bursitis": "التهاب الجراب",
    "callus": "كالو/ثَفَن",
    "carbon monoxide poisoning": "تسمّم أول أكسيد الكربون",
    "cardiac arrest": "توقف القلب",
    "cardiomyopathy": "اعتلال عضلة القلب",
    "carpal tunnel syndrome": "متلازمة النفق الرسغي",
    "cat scratch disease": "مرض خدش القطة",
    "cataract": "إعتام عدسة العين (مياه بيضاء)",
    "celiac disease": "الداء البطني (السيلياك)",
    "cellulitis or abscess of mouth": "التهاب نسيج خلوي أو خراج بالفم",
    "central atherosclerosis": "تصلّب شرايين مركزي",
    "central retinal artery or vein occlusion": "انسداد الشريان أو الوريد الشبكي المركزي",
    "cerebral edema": "وذمة دماغية",
    "cerebral palsy": "شلل دماغي",
    "cervical cancer": "سرطان عنق الرحم",
    "cervical disorder": "اضطراب عنق الرحم",
    "cervicitis": "التهاب عنق الرحم",
    "chalazion": "كيس دهني بالجفن (بردة)",
    "chickenpox": "جدري الماء",
    "chlamydia": "الكلاميديا",
    "cholecystitis": "التهاب المرارة",
    "choledocholithiasis": "حصوات القناة الصفراوية المشتركة",
    "cholesteatoma": "ورم كوليستياتومي بالأذن",
    "chondromalacia of the patella": "تلين غضروف الرضفة",
    "chorioretinitis": "التهاب المشيمية والشبكية",
    "chronic back pain": "ألم ظهر مزمن",
    "chronic constipation": "إمساك مزمن",
    "chronic glaucoma": "زَرَق مزمن",
    "chronic inflammatory demyelinating polyneuropathy (cidp)": "اعتلال الأعصاب الالتهابي المزمن المزيل للميالين",
    "chronic kidney disease": "مرض كلوي مزمن",
    "chronic knee pain": "ألم ركبة مزمن",
    "chronic obstructive pulmonary disease (copd)": "مرض الانسداد الرئوي المزمن",
    "chronic otitis media": "التهاب الأذن الوسطى المزمن",
    "chronic pain disorder": "اضطراب ألم مزمن",
    "chronic pancreatitis": "التهاب البنكرياس المزمن",
    "chronic rheumatic fever": "حمّى روماتيزمية مزمنة",
    "chronic sinusitis": "التهاب الجيوب الأنفية المزمن",
    "cirrhosis": "تشمع/تليف الكبد",
    "coagulation (bleeding) disorder": "اضطراب تخثّر/نزف",
    "cold sore": "قرحة برد (هربس شفوي)",
    "colonic polyp": "سليلة/بوليب بالقولون",
    "colorectal cancer": "سرطان القولون والمستقيم",
    "common cold": "نزلة برد",
    "complex regional pain syndrome": "متلازمة الألم الناحي المعقّد",
    "concussion": "ارتجاج",
    "conduct disorder": "اضطراب السلوك",
    "conductive hearing loss": "فقدان سمع توصيلي",
    "congenital heart defect": "عيب خلقي بالقلب",
    "conjunctivitis": "التهاب الملتحمة",
    "conjunctivitis due to allergy": "التهاب ملتحمة تحسّسي",
    "conjunctivitis due to bacteria": "التهاب ملتحمة بكتيري",
    "conjunctivitis due to virus": "التهاب ملتحمة فيروسي",
    "contact dermatitis": "التهاب جلد تماسي",
    "cornea infection": "عدوى القرنية",
    "corneal abrasion": "سحج/خدش القرنية",
    "corneal disorder": "اضطراب القرنية",
    "coronary atherosclerosis": "تصلّب الشرايين التاجية",
    "cranial nerve palsy": "شلل أحد الأعصاب القحفية",
    "crohn disease": "داء كرون",
    "croup": "الخُناق (التهاب الحنجرة والقصبة عند الأطفال)",
    "crushing injury": "إصابة/سَحْق",
    "cryptorchidism": "خصية مُعلّقة (عدم نزول الخصية)",
    "cushing syndrome": "متلازمة كوشينغ",
    "cysticercosis": "داء الكيسات المذنّبة",
    "cystitis": "التهاب المثانة",
    "de quervain disease": "التهاب غمد الأوتار (دي كيرفان)",
    "deep vein thrombosis (dvt)": "جلطة/خثار وريدي عميق",
    "degenerative disc disease": "تنكّس الأقراص بين الفقرات",
    "delirium": "هذيان",
    "dementia": "خرف",
    "dental caries": "تسوّس الأسنان",
    "depression": "اكتئاب",
    "dermatitis due to sun exposure": "التهاب جلد بسبب الشمس",
    "developmental disability": "إعاقة نمائية",
    "deviated nasal septum": "انحراف الحاجز الأنفي",
    "diabetes insipidus": "السكري الكاذب",
    "diabetic ketoacidosis": "الحماض الكيتوني السكري",
    "diabetic peripheral neuropathy": "اعتلال الأعصاب الطرفية السكري",
    "diabetic retinopathy": "اعتلال الشبكية السكري",
    "diaper rash": "طفح الحفاض",
    "dislocation of the finger": "خلع الإصبع",
    "dislocation of the foot": "خلع القدم",
    "dislocation of the knee": "خلع الركبة",
    "dislocation of the patella": "خلع الرضفة",
    "dislocation of the shoulder": "خلع الكتف",
    "dislocation of the vertebra": "خلع فقرة",
    "dislocation of the wrist": "خلع الرسغ",
    "dissociative disorder": "اضطراب تفارقي",
    "diverticulitis": "التهاب الرتوج",
    "diverticulosis": "داء الرتوج",
    "down syndrome": "متلازمة داون",
    "drug abuse": "إساءة استخدام المخدرات",
    "drug abuse (barbiturates)": "إساءة استخدام الباربيتورات",
    "drug abuse (cocaine)": "إساءة استخدام الكوكايين",
    "drug abuse (methamphetamine)": "إساءة استخدام الميثامفيتامين",
    "drug abuse (opioids)": "إساءة استخدام الأفيونات",
    "drug poisoning due to medication": "تسمّم دوائي بسبب الأدوية",
    "drug reaction": "تفاعل/تحسّس دوائي",
    "drug withdrawal": "أعراض انسحاب المخدرات",
    "dry eye of unknown cause": "جفاف العين (سبب غير معروف)",
    "dyshidrosis": "إكزيما حويصلية (تعَرُّق حُويصلي)",
    "dysthymic disorder": "اضطراب عُسر المزاج",
    "ear drum damage": "تلف/ضرر طبلة الأذن",
    "ear wax impaction": "انحشار شمع الأذن",
    "eating disorder": "اضطراب الأكل",
    "ectopic pregnancy": "حمل خارج الرحم",
    "ectropion": "شتر خارجي للجفن",
    "eczema": "إكزيما",
    "emphysema": "انتفاخ الرئة",
    "empyema": "دُبيلة (صديد بالجنب/البلورا)",
    "encephalitis": "التهاب الدماغ",
    "endocarditis": "التهاب شِغاف القلب",
    "endometrial cancer": "سرطان بطانة الرحم",
    "endometriosis": "بطانة الرحم المهاجرة",
    "endophthalmitis": "التهاب باطن المقلة",
    "envenomation from spider or animal bite": "تسمّم/لدغ من عنكبوت أو عضة حيوان",
    "ependymoma": "ورم بطاني عصبي",
    "epididymitis": "التهاب البربخ",
    "epidural hemorrhage": "نزف فوق الجافية",
    "epilepsy": "صرع",
    "erectile dysfunction": "ضعف الانتصاب",
    "erythema multiforme": "الحمامى عديدة الأشكال",
    "esophageal cancer": "سرطان المريء",
    "esophageal varices": "دوالي المريء",
    "esophagitis": "التهاب المريء",
    "essential tremor": "رجفة/رعاش أساسي",
    "eustachian tube dysfunction (ear disorder)": "خلل قناة استاكيوس",
    "eye alignment disorder": "اضطراب محاذاة العين (حول)",
    "factitious disorder": "اضطراب مُفتعل",
    "female genitalia infection": "عدوى بالأعضاء التناسلية الأنثوية",
    "fibroadenoma": "ورم ليفي غُدي",
    "fibromyalgia": "الألم العضلي الليفي",
    "flat feet": "تسطّح القدمين",
    "floaters": "عوائم العين",
    "flu": "إنفلونزا",
    "fluid overload": "زيادة/احتباس السوائل",
    "folate deficiency": "نقص حمض الفوليك",
    "food allergy": "حساسية الطعام",
    "foreign body in the eye": "جسم غريب في العين",
    "foreign body in the gastrointestinal tract": "جسم غريب في الجهاز الهضمي",
    "foreign body in the throat": "جسم غريب في الحلق",
    "fracture of the ankle": "كسر الكاحل",
    "fracture of the arm": "كسر الذراع",
    "fracture of the facial bones": "كسر عظام الوجه",
    "fracture of the finger": "كسر الإصبع",
    "fracture of the foot": "كسر القدم",
    "fracture of the hand": "كسر اليد",
    "fracture of the jaw": "كسر الفك",
    "fracture of the leg": "كسر الساق",
    "fracture of the neck": "كسر/إصابة بالرقبة",
    "fracture of the patella": "كسر الرضفة",
    "fracture of the pelvis": "كسر الحوض",
    "fracture of the rib": "كسر ضلع",
    "fracture of the shoulder": "كسر الكتف",
    "fracture of the skull": "كسر الجمجمة",
    "fracture of the vertebra": "كسر فقرة",
    "friedrich ataxia": "رنح فريدريخ",
    "fungal infection of the hair": "عدوى فطرية بالشعر",
    "fungal infection of the skin": "عدوى فطرية بالجلد",
    "gallstone": "حصوة مرارية",
    "ganglion cyst": "كيس زُلالي (غانغليون)",
    "gastritis": "التهاب المعدة",
    "gastroduodenal ulcer": "قرحة معدية/اثنا عشرية",
    "gastroesophageal reflux disease (gerd)": "الارتجاع المعدي المريئي",
    "gastrointestinal hemorrhage": "نزيف بالجهاز الهضمي",
    "gastroparesis": "خزل المعدة",
    "genital herpes": "هربس تناسلي",
    "gestational diabetes": "سكري الحمل",
    "glaucoma": "الزَرَق (المياه الزرقاء)",
    "goiter": "تضخّم الغدة الدرقية (جويتر)",
    "gonorrhea": "السيلان",
    "gout": "النقرس",
    "graves disease": "داء غريفز",
    "guillain barre syndrome": "متلازمة غيلان باريه",
    "gum disease": "مرض/التهاب اللثة",
    "gynecomastia": "تثدّي الرجال",
    "hammer toe": "إصبع مطرقي",
    "hashimoto thyroiditis": "التهاب الدرقية لهاشيموتو",
    "head and neck cancer": "سرطان الرأس والرقبة",
    "head injury": "إصابة الرأس",
    "headache after lumbar puncture": "صداع بعد البزل القطني",
    "heart attack": "نوبة قلبية (احتشاء عضلة القلب)",
    "heart block": "حصار قلبي",
    "heart contusion": "كدمة القلب",
    "heart failure": "فشل/قصور القلب",
    "heat exhaustion": "إجهاد حراري",
    "hemangioma": "ورم وعائي دموي",
    "hemarthrosis": "نزف داخل المفصل",
    "hematoma": "ورم دموي",
    "hemiplegia": "شلل نصفي",
    "hemorrhoids": "بواسير",
    "hepatitis due to a toxin": "التهاب كبد سُمّي",
    "herniated disk": "انزلاق/فتق غضروفي",
    "herpangina": "هربانجينا (التهاب حلق فيروسي)",
    "hiatal hernia": "فتق حجابي",
    "hidradenitis suppurativa": "التهاب الغدد العرقية القيحي",
    "high blood pressure": "ارتفاع ضغط الدم",
    "hirschsprung disease": "داء هيرشسبرونغ",
    "hormone disorder": "اضطراب هرموني",
    "hpv": "فيروس الورم الحليمي البشري",
    "hydrocele of the testicle": "قيلة مائية بالخصية",
    "hydrocephalus": "استسقاء دماغي",
    "hydronephrosis": "استسقاء/موه الكلية",
    "hypercholesterolemia": "ارتفاع كوليسترول الدم",
    "hyperemesis gravidarum": "قيء حملي مفرط",
    "hypergammaglobulinemia": "فرط غاماغلوبولين الدم",
    "hyperkalemia": "ارتفاع بوتاسيوم الدم",
    "hypernatremia": "ارتفاع صوديوم الدم",
    "hyperopia": "طول النظر",
    "hypertension of pregnancy": "ارتفاع ضغط الدم أثناء الحمل",
    "hypertensive heart disease": "مرض القلب بسبب ارتفاع الضغط",
    "hypocalcemia": "نقص كالسيوم الدم",
    "hypoglycemia": "نقص سكر الدم",
    "hypokalemia": "نقص بوتاسيوم الدم",
    "hyponatremia": "نقص صوديوم الدم",
    "hypothyroidism": "قصور الغدة الدرقية",
    "hypovolemia": "نقص حجم الدم",
    "idiopathic absence of menstruation": "انقطاع طمث مجهول السبب",
    "idiopathic excessive menstruation": "غزارة طمث مجهولة السبب",
    "idiopathic infrequent menstruation": "قلّة الطمث مجهولة السبب",
    "idiopathic irregular menstrual cycle": "عدم انتظام الدورة مجهول السبب",
    "idiopathic nonmenstrual bleeding": "نزف غير طمثي مجهول السبب",
    "idiopathic painful menstruation": "عسر طمث مجهول السبب",
    "ileus": "علوص (شلل الأمعاء)",
    "impetigo": "القوباء",
    "impulse control disorder": "اضطراب التحكم بالاندفاع",
    "indigestion": "عسر هضم",
    "induced abortion": "إجهاض مُحرَّض",
    "infectious gastroenteritis": "التهاب معدة وأمعاء معدي",
    "ingrown toe nail": "ظفر نامٍ",
    "inguinal hernia": "فتق إربي",
    "injury of the ankle": "إصابة الكاحل",
    "injury to internal organ": "إصابة عضو داخلي",
    "injury to the abdomen": "إصابة البطن",
    "injury to the arm": "إصابة الذراع",
    "injury to the face": "إصابة الوجه",
    "injury to the hand": "إصابة اليد",
    "injury to the knee": "إصابة الركبة",
    "injury to the leg": "إصابة الساق",
    "injury to the shoulder": "إصابة الكتف",
    "injury to the spinal cord": "إصابة الحبل الشوكي",
    "injury to the trunk": "إصابة الجذع",
    "insect bite": "لدغة/قرصة حشرة",
    "insulin overdose": "جرعة زائدة من الإنسولين",
    "interstitial lung disease": "مرض رئوي خلالي",
    "intertrigo (skin condition)": "التهاب ثنايا الجلد (سُحاج)",
    "intestinal cancer": "سرطان الأمعاء",
    "intestinal disease": "مرض بالأمعاء",
    "intestinal malabsorption": "سوء امتصاص معوي",
    "intestinal obstruction": "انسداد الأمعاء",
    "intracerebral hemorrhage": "نزيف داخل الدماغ",
    "intracranial abscess": "خراج داخل القحف",
    "intracranial hemorrhage": "نزيف داخل القحف",
    "iridocyclitis": "التهاب القزحية والجسم الهدبي",
    "iron deficiency anemia": "فقر دم بنقص الحديد",
    "irritable bowel syndrome": "متلازمة القولون العصبي",
    "ischemia of the bowel": "نقص تروية الأمعاء",
    "ischemic heart disease": "مرض القلب الإقفاري",
    "itching of unknown cause": "حكّة مجهولة السبب",
    "jaw disorder": "اضطراب الفك",
    "joint effusion": "انصباب مفصلي",
    "juvenile rheumatoid arthritis": "التهاب مفاصل روماتويدي يافع",
    "kaposi sarcoma": "ساركوما كابوزي",
    "kidney cancer": "سرطان الكلى",
    "kidney failure": "فشل كلوي",
    "kidney stone": "حصوة كلى",
    "knee ligament or meniscus tear": "تمزق أربطة الركبة أو الغضروف الهلالي",
    "labyrinthitis": "التهاب التيه (الأذن الداخلية)",
    "lactose intolerance": "عدم تحمّل اللاكتوز",
    "laryngitis": "التهاب الحنجرة",
    "lateral epicondylitis (tennis elbow)": "التهاب اللقيمة الوحشية (مرفق التنس)",
    "leukemia": "ابيضاض الدم (لوكيميا)",
    "lewy body dementia": "خرف أجسام ليوي",
    "lice": "قمل",
    "lichen planus": "الحزاز المسطّح",
    "lichen simplex": "حزاز بسيط مزمن",
    "lipoma": "ورم شحمي",
    "liver cancer": "سرطان الكبد",
    "liver disease": "مرض كبدي",
    "lumbago": "ألم أسفل الظهر (لومباغو)",
    "lung cancer": "سرطان الرئة",
    "lung contusion": "كدمة الرئة",
    "lyme disease": "داء لايم",
    "lymphadenitis": "التهاب العقد اللمفاوية",
    "lymphangitis": "التهاب الأوعية اللمفاوية",
    "lymphedema": "وذمة لمفاوية",
    "lymphogranuloma venereum": "الورم الحبيبي اللمفي المنقول جنسيًا",
    "lymphoma": "لمفوما",
    "macular degeneration": "تنكّس البقعة الصفراء",
    "magnesium deficiency": "نقص المغنيسيوم",
    "male genitalia infection": "عدوى بالأعضاء التناسلية الذكرية",
    "malignant hypertension": "ارتفاع ضغط خبيث",
    "marijuana abuse": "إساءة استخدام الماريجوانا",
    "mastoiditis": "التهاب الخشاء",
    "melanoma": "ميلانوما (سرطان الجلد الخبيث)",
    "meniere disease": "داء منيير",
    "meningioma": "ورم سحائي",
    "meningitis": "التهاب السحايا",
    "menopause": "انقطاع الطمث (سن اليأس)",
    "metastatic cancer": "سرطان منتشر (نقائلي)",
    "migraine": "صداع نصفي (شقيقة)",
    "missed abortion": "إجهاض فائت",
    "mitral valve disease": "مرض الصمام المترالي",
    "mittelschmerz": "ألم الإباضة",
    "molluscum contagiosum": "المليساء المعدية",
    "mononeuritis": "التهاب عصب مفرد",
    "mononucleosis": "داء الوحيدات (مونو)",
    "mucositis": "التهاب الغشاء المخاطي",
    "multiple myeloma": "الورم النقوي المتعدد",
    "multiple sclerosis": "التصلّب المتعدد",
    "mumps": "نكاف",
    "muscle spasm": "تشنّج عضلي",
    "muscular dystrophy": "حثل/ضمور عضلي",
    "myasthenia gravis": "الوهن العضلي الوبيل",
    "myocarditis": "التهاب عضلة القلب",
    "myopia": "قصر النظر",
    "myositis": "التهاب العضلات",
    "narcolepsy": "نوم قهري",
    "nasal polyp": "لحمية/سليلة أنفية",
    "necrotizing fasciitis": "التهاب لفافة ناخر",
    "neonatal jaundice": "يرقان حديثي الولادة",
    "nerve impingement near the shoulder": "انضغاط عصب قرب الكتف",
    "neuralgia": "ألم عصبي",
    "neuropathy due to drugs": "اعتلال أعصاب بسبب الأدوية",
    "neurosis": "عُصاب",
    "noninfectious gastroenteritis": "التهاب معدة وأمعاء غير معدي",
    "normal pressure hydrocephalus": "استسقاء دماغي بضغط طبيعي",
    "nose disorder": "اضطراب الأنف",
    "obsessive compulsive disorder (ocd)": "اضطراب الوسواس القهري",
    "obstructive sleep apnea (osa)": "انقطاع النفس الانسدادي أثناء النوم",
    "omphalitis": "التهاب السرة",
    "onychomycosis": "فطريات الأظافر",
    "open wound from surgical incision": "جرح مفتوح من شق جراحي",
    "open wound of the abdomen": "جرح مفتوح بالبطن",
    "open wound of the ear": "جرح مفتوح بالأذن",
    "open wound of the eye": "جرح مفتوح بالعين",
    "open wound of the finger": "جرح مفتوح بالإصبع",
    "open wound of the hand": "جرح مفتوح باليد",
    "open wound of the knee": "جرح مفتوح بالركبة",
    "open wound of the lip": "جرح مفتوح بالشفة",
    "open wound of the mouth": "جرح مفتوح بالفم",
    "open wound of the neck": "جرح مفتوح بالرقبة",
    "open wound of the nose": "جرح مفتوح بالأنف",
    "oppositional disorder": "اضطراب العناد التحدّي",
    "optic neuritis": "التهاب العصب البصري",
    "oral mucosal lesion": "آفة بالغشاء المخاطي الفموي",
    "oral thrush (yeast infection)": "القلاع الفموي (فطريات)",
    "orbital cellulitis": "التهاب نسيج خلوي بالحجاج",
    "orthostatic hypotension": "هبوط ضغط انتصابي",
    "osteoarthritis": "الفصال العظمي (خشونة المفاصل)",
    "osteochondroma": "ورم عظمي غضروفي",
    "osteochondrosis": "تنخّر/اعتلال عظمي غضروفي",
    "osteomyelitis": "التهاب العظم والنقي",
    "osteoporosis": "هشاشة العظام",
    "otitis externa (swimmer's ear)": "التهاب الأذن الخارجية",
    "otitis media": "التهاب الأذن الوسطى",
    "otosclerosis": "تصلّب الأذن",
    "ovarian cancer": "سرطان المبيض",
    "ovarian cyst": "كيس مبيضي",
    "ovarian torsion": "التواء المبيض",
    "overflow incontinence": "سلس فيضاني",
    "pain after an operation": "ألم بعد عملية جراحية",
    "pain disorder affecting the neck": "اضطراب ألم يؤثر على الرقبة",
    "pancreatic cancer": "سرطان البنكرياس",
    "panic attack": "نوبة هلع",
    "panic disorder": "اضطراب الهلع",
    "parasitic disease": "مرض طفيلي",
    "parathyroid adenoma": "ورم غُدي جار درقي",
    "parkinson disease": "مرض باركنسون",
    "paronychia": "التهاب حول الظفر",
    "paroxysmal supraventricular tachycardia": "تسرّع فوق بطيني انتيابي",
    "paroxysmal ventricular tachycardia": "تسرّع بطيني انتيابي",
    "pelvic inflammatory disease": "التهاب الحوض",
    "pelvic organ prolapse": "هبوط أعضاء الحوض",
    "pericarditis": "التهاب التامور",
    "peripheral arterial disease": "مرض الشرايين الطرفية",
    "peripheral arterial embolism": "صِمّة شريانية طرفية",
    "peripheral nerve disorder": "اضطراب الأعصاب الطرفية",
    "perirectal infection": "عدوى حول المستقيم",
    "peritonitis": "التهاب الصفاق (البريتون)",
    "peritonsillar abscess": "خراج حول اللوزة",
    "persistent vomiting of unknown cause": "قيء مستمر مجهول السبب",
    "personality disorder": "اضطراب الشخصية",
    "peyronie disease": "داء بيروني",
    "phimosis": "تضيّق القلفة",
    "pilonidal cyst": "كيس/ناسور شعري",
    "pinworm infection": "عدوى الدودة الدبوسية",
    "pituitary adenoma": "ورم غُدي نخامي",
    "pituitary disorder": "اضطراب الغدة النخامية",
    "pityriasis rosea": "النخالية الوردية",
    "placental abruption": "انفصال المشيمة",
    "plantar fasciitis": "التهاب اللفافة الأخمصية",
    "pleural effusion": "انصباب جنبي",
    "pneumoconiosis": "داء الرئة الغباري",
    "pneumonia": "التهاب رئوي",
    "pneumothorax": "استرواح الصدر",
    "poisoning due to analgesics": "تسمّم بسبب المسكنات",
    "poisoning due to antidepressants": "تسمّم بسبب مضادات الاكتئاب",
    "poisoning due to antimicrobial drugs": "تسمّم بسبب مضادات الميكروبات",
    "poisoning due to ethylene glycol": "تسمّم بالإيثيلين غلايكول",
    "poisoning due to gas": "تسمّم بالغاز",
    "poisoning due to opioids": "تسمّم بالأفيونات",
    "poisoning due to sedatives": "تسمّم بالمهدئات",
    "polycystic kidney disease": "مرض الكلى متعددة الكيسات",
    "polycystic ovarian syndrome (pcos)": "متلازمة تكيس المبايض",
    "polycythemia vera": "كثرة الحُمر الحقيقية",
    "polymyalgia rheumatica": "ألم عضلي روماتيزمي",
    "post-traumatic stress disorder (ptsd)": "اضطراب ما بعد الصدمة",
    "postoperative infection": "عدوى بعد الجراحة",
    "postpartum depression": "اكتئاب ما بعد الولادة",
    "preeclampsia": "ما قبل تسمّم الحمل",
    "pregnancy": "حمل",
    "premature atrial contractions (pacs)": "انقباضات أذينية مبكرة",
    "premature ovarian failure": "فشل مبيضي مبكر",
    "premature rupture of amniotic membrane": "تمزّق مبكر للأغشية الأمنيوسية",
    "premature ventricular contractions (pvcs)": "انقباضات بطينية مبكرة",
    "premenstrual tension syndrome": "متلازمة ما قبل الحيض",
    "presbyacusis": "ضعف سمع شيخوخي",
    "presbyopia": "طول نظر شيخوخي",
    "primary immunodeficiency": "نقص مناعة أولي",
    "primary insomnia": "أرق أولي",
    "primary kidney disease": "مرض كلوي أولي",
    "primary thrombocythemia": "كثرة الصفيحات الأولية",
    "problem during pregnancy": "مشكلة أثناء الحمل",
    "prostate cancer": "سرطان البروستاتا",
    "prostatitis": "التهاب البروستاتا",
    "protein deficiency": "نقص البروتين",
    "pseudotumor cerebri": "ورم كاذب بالدماغ (ارتفاع ضغط داخل القحف الحميد)",
    "psoriasis": "صدفية",
    "psychosexual disorder": "اضطراب نفسي-جنسي",
    "psychotic disorder": "اضطراب ذهاني",
    "pterygium": "ظُفرة",
    "pulmonary congestion": "احتقان رئوي",
    "pulmonary embolism": "انسداد/صِمّة رئوية",
    "pulmonary eosinophilia": "فرط حمضات رئوي",
    "pulmonary fibrosis": "تليّف رئوي",
    "pulmonary hypertension": "ارتفاع ضغط الشريان الرئوي",
    "pyelonephritis": "التهاب حويضة وكلية",
    "pyloric stenosis": "تضيّق البواب",
    "pyogenic skin infection": "عدوى جلدية قيحية",
    "rectal disorder": "اضطراب المستقيم",
    "restless leg syndrome": "متلازمة تململ الساقين",
    "retinal detachment": "انفصال الشبكية",
    "retinopathy due to high blood pressure": "اعتلال الشبكية بسبب الضغط",
    "rheumatoid arthritis": "التهاب مفاصل روماتويدي",
    "rocky mountain spotted fever": "حمّى جبال روكي المبقّعة",
    "rosacea": "الوردية",
    "rotator cuff injury": "إصابة الكُفّة المُدوّرة بالكتف",
    "salivary gland disorder": "اضطراب الغدد اللعابية",
    "sarcoidosis": "الساركويد",
    "scabies": "جرب",
    "scar": "ندبة",
    "scarlet fever": "الحمّى القرمزية",
    "schizophrenia": "فُصام",
    "sciatica": "عرق النسا",
    "scleritis": "التهاب الصلبة",
    "scleroderma": "تصلّب الجلد",
    "scoliosis": "جنف",
    "seasonal allergies (hay fever)": "حساسية موسمية (حمّى القش)",
    "sebaceous cyst": "كيس دهني",
    "seborrheic dermatitis": "التهاب الجلد الدهني",
    "seborrheic keratosis": "تقرّن دهني",
    "sensorineural hearing loss": "فقدان سمع عصبي-حسي",
    "sepsis": "إنتان/تعفّن الدم",
    "septic arthritis": "التهاب مفصل إنتاني",
    "shingles (herpes zoster)": "الحزام الناري",
    "sialoadenitis": "التهاب الغدة اللعابية",
    "sick sinus syndrome": "متلازمة الجيب المريض",
    "sickle cell anemia": "فقر الدم المنجلي",
    "sickle cell crisis": "أزمة منجلية",
    "sinus bradycardia": "بطء قلب جيبي",
    "sjogren syndrome": "متلازمة شوغرن",
    "skin cancer": "سرطان الجلد",
    "skin disorder": "اضطراب جلدي",
    "skin pigmentation disorder": "اضطراب تصبّغ الجلد",
    "skin polyp": "زائدة جلدية",
    "smoking or tobacco addiction": "إدمان التدخين/التبغ",
    "social phobia": "رهاب اجتماعي",
    "soft tissue sarcoma": "ساركوما الأنسجة الرخوة",
    "somatization disorder": "اضطراب التحوّل/الجسدنة",
    "spermatocele": "قيلة منوية",
    "spina bifida": "السنسنة المشقوقة",
    "spinal stenosis": "تضيّق القناة الشوكية",
    "spinocerebellar ataxia": "رنح نخاعي مخيخي",
    "spondylitis": "التهاب الفقار",
    "spondylolisthesis": "انزلاق فقاري",
    "spondylosis": "تنكّس فقاري",
    "spontaneous abortion": "إجهاض تلقائي",
    "sprain or strain": "التواء أو شدّ عضلي",
    "stenosis of the tear duct": "تضيّق القناة الدمعية",
    "stomach cancer": "سرطان المعدة",
    "strep throat": "التهاب الحلق بالعقديات",
    "stress incontinence": "سلس إجهادي",
    "stricture of the esophagus": "تضيّق المريء",
    "stroke": "سكتة دماغية",
    "stye": "دُمّل/شُعيرة بالجفن",
    "subarachnoid hemorrhage": "نزف تحت العنكبوتية",
    "subconjunctival hemorrhage": "نزف تحت الملتحمة",
    "subdural hemorrhage": "نزف تحت الجافية",
    "substance-related mental disorder": "اضطراب نفسي مرتبط بالمواد",
    "syphilis": "الزهري",
    "systemic lupus erythematosis (sle)": "الذئبة الحمراء الجهازية",
    "teething syndrome": "متلازمة التسنين",
    "temporary or benign blood in urine": "دم بالبول مؤقت/حميد",
    "temporomandibular joint disorder": "اضطراب مفصل الفك الصدغي",
    "tendinitis": "التهاب الأوتار",
    "tension headache": "صداع توتري",
    "testicular disorder": "اضطراب الخصية",
    "testicular torsion": "التواء الخصية",
    "thalassemia": "الثلاسيميا",
    "thoracic aortic aneurysm": "تمدد الشريان الأورطي الصدري",
    "thoracic outlet syndrome": "متلازمة مخرج الصدر",
    "threatened pregnancy": "تهديد بالإجهاض (حمل مهدد)",
    "thrombocytopenia": "نقص الصفائح الدموية",
    "thrombophlebitis": "التهاب الوريد الخثاري",
    "thyroid disease": "مرض الغدة الدرقية",
    "thyroid nodule": "عُقيدة درقية",
    "tic (movement) disorder": "اضطراب العرّات (تيك)",
    "tietze syndrome": "متلازمة تيتزه",
    "tinnitus of unknown cause": "طنين مجهول السبب",
    "tonsillar hypertrophy": "تضخّم اللوزتين",
    "tonsillitis": "التهاب اللوزتين",
    "tooth abscess": "خراج سنّي",
    "tooth disorder": "اضطراب الأسنان",
    "torticollis": "صعر (تيبّس/التواء الرقبة)",
    "tourette syndrome": "متلازمة توريت",
    "toxic multinodular goiter": "تضخّم درقي عقدي سُمّي",
    "toxoplasmosis": "داء المقوسات (توكسوبلازما)",
    "tracheitis": "التهاب القصبة الهوائية",
    "transient ischemic attack": "نوبة إقفارية عابرة",
    "trichomonas infection": "عدوى المشعّرات",
    "trigeminal neuralgia": "ألم العصب ثلاثي التوائم",
    "trigger finger (finger disorder)": "إصبع زنّاد",
    "tuberous sclerosis": "التصلّب الحدبي",
    "turner syndrome": "متلازمة تيرنر",
    "ulcerative colitis": "التهاب القولون التقرّحي",
    "urethral disorder": "اضطراب الإحليل",
    "urethral stricture": "تضيّق الإحليل",
    "urethral valves": "صمامات الإحليل",
    "urethritis": "التهاب الإحليل",
    "urge incontinence": "سلس إلحاحي",
    "urinary tract infection": "التهاب المسالك البولية",
    "urinary tract obstruction": "انسداد المسالك البولية",
    "uterine atony": "وهن/ارتخاء الرحم",
    "uterine fibroids": "أورام ليفية بالرحم",
    "uveitis": "التهاب العنبية",
    "vacterl syndrome": "متلازمة فاكتيرل",
    "vaginal cyst": "كيس مهبلي",
    "vaginal yeast infection": "عدوى فطرية مهبلية",
    "vaginismus": "تشنّج/مهبلية (تقبّض مهبلي)",
    "vaginitis": "التهاب المهبل",
    "varicocele of the testicles": "دوالي الخصية",
    "varicose veins": "دوالي الساقين",
    "vasculitis": "التهاب الأوعية الدموية",
    "venous insufficiency": "قصور وريدي",
    "vertebrobasilar insufficiency": "قصور فقري قاعدي",
    "vesicoureteral reflux": "ارتجاع مثاني حالبي",
    "viral exanthem": "طفح فيروسي",
    "viral warts": "ثآليل فيروسية",
    "vitamin a deficiency": "نقص فيتامين A",
    "vitamin b12 deficiency": "نقص فيتامين B12",
    "vitamin d deficiency": "نقص فيتامين D",
    "vitreous degeneration": "تنكّس الجسم الزجاجي",
    "vitreous hemorrhage": "نزيف الجسم الزجاجي",
    "vocal cord polyp": "سلّيلة/بوليب بالحبل الصوتي",
    "volvulus": "التواء الأمعاء",
    "vulvar disorder": "اضطراب الفرج",
    "vulvodynia": "ألم الفرج",
    "wernicke korsakoff syndrome": "متلازمة فيرنيكه-كورساكوف",
    "white blood cell disease": "مرض كريات الدم البيضاء",
    "whooping cough": "السعال الديكي",
    "wilson disease": "داء ويلسون",
    "yeast infection": "عدوى فطرية (كانديدا)",
}

    def translate_diseases_in_response(text: str) -> str:
      for eng, ar in disease_translation.items():
        text = re.sub(r'\b' + re.escape(eng) + r'\b', ar, text, flags=re.IGNORECASE)
        return text

    final = response.strip()
    finaL = translate_diseases_in_response(final)

    if specialty:
        finaL += f"\n\n👨‍⚕️ يمكنك استشارة طبيباً متخصصاً في: {specialty}"
    else:
      finaL += "\n\n👨‍⚕️ التخصص المقترح: باطنة"

    finaL += "\n(هذا اقتراح مبدئي فقط)"
    return finaL

In [11]:
if __name__ == "__main__":
    while True:
        symptoms = input("\nاكتب الأعراض (أو اكتب exit للخروج): ").strip()
        if symptoms.lower() in ["exit", "خروج", "قف"]:
            break
        if not symptoms:
            continue

        print("\nجاري التفكير...\n")
        answer = ask_medical_ai(symptoms)
        print(answer)
        print("-" * 70)


اكتب الأعراض (أو اكتب exit للخروج): ألم أسفل البطن، نزيف مهبلي غير طبيعي، ألم في الحوض

جاري التفكير...

🔎 تحليل الأعراض:
الأعراض المذكورة قد تشير إلى أكثر من حالة طبية.

🦠 أمراض محتملة:
1- pelvic inflammatory disease: يتوافق مع مجموعة من الأعراض المذكورة.
2- ovarian cyst: يتوافق مع مجموعة من الأعراض المذكورة.
3- ectopic pregnancy: يتوافق مع مجموعة من الأعراض المذكورة.

❓ معلومات إضافية قد تساعد:
1) منذ متى بدأت الأعراض؟ وهل تزداد شدتها؟
2) هل توجد أعراض خطيرة مثل ألم صدر شديد أو إغماء؟
3) هل توجد أعراض خطيرة مثل ضيق تنفس شديد أو ألم صدر شديد؟

⚠️ تنبيه:
هذه الحالة لربما تكون طارئة وتستدعي تدخلاً طبياً فورياً. من فضلك توجه لأقرب مستشفى فوراً، أو اتصل بلإسعاف:

👨‍⚕️ يمكنك استشارة طبيباً متخصصاً في: جهاز هضمي
(هذا اقتراح مبدئي فقط)
----------------------------------------------------------------------

اكتب الأعراض (أو اكتب exit للخروج): exit
